# 03 - Degrade and augment (CCTV-style, GPU)

Turns the **undegraded** images in `CLEAN_DIR` into the **degraded** training set in
`data/synthetic_degraded/{class}/`, simulating restaurant-camera footage. **Novelty #1.**

- **Runs on the GPU**, batched, with pure PyTorch (`torch.nn.functional`) - no new packages. Falls back to CPU.
- **Class-agnostic & de-biased:** the degradation never sees the label, so no effect can correlate with the class.
- **All tunable knobs are in one place:** the **`DEGRADE` settings** cell below. Change a number there to make an
  effect stronger/weaker (e.g. if images come out too bright). Each effect is then its own small function.
- **Reproducible** (one fixed seed). JPEG is applied at save. No timestamp/"REC" HUD is ever drawn.

## Configuration + paths

In [ ]:
import os, math, random, shutil
# --- choose the GPU here (no terminal needed; just press Run) ---
# Set BEFORE importing torch. Change "0" if GPU 0 is busy, then restart the kernel.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

import sys
from pathlib import Path
# Make the local shlomi/utils.py importable whether launched from shlomi/ or the repo root.
for _cand in (Path.cwd(), Path.cwd() / "shlomi"):
    if (_cand / "utils.py").exists() and str(_cand) not in sys.path:
        sys.path.insert(0, str(_cand)); break
import utils

torch.set_grad_enabled(False)                       # inference-only preprocessing
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

INPUT_DIR  = utils.CLEAN_DIR          # undegraded images, per-class subfolders (e.g. Diff_DataSet)
OUTPUT_DIR = utils.DEGRADED_DIR       # data/synthetic_degraded/{class}/
BATCH = 64                            # images per GPU batch
SIZE = 224                            # working resolution
NUM_AUG_CHOICES = [2, 3, 4, 5]        # how many degraded copies to make per source image

CLASSES = sorted([d.name for d in INPUT_DIR.iterdir() if d.is_dir()]) if INPUT_DIR.is_dir() else []
print("device :", DEVICE)
print("input  :", INPUT_DIR)
print("output :", OUTPUT_DIR)
print("classes:", CLASSES)

def reset_degraded_folder():
    if OUTPUT_DIR.exists():
        print("Deleting existing degraded folder..."); shutil.rmtree(OUTPUT_DIR)
    for cls in CLASSES:
        os.makedirs(OUTPUT_DIR / cls, exist_ok=True)
    print("Fresh degraded folder ready:", OUTPUT_DIR)

## Degradation settings - TUNE HERE

Every knob for every effect is in the `DEGRADE` dict below. Each effect has a **`prob`** (how often it's
applied, 0-1) and a **strength range** (a `(low, high)` pair sampled per image).

**Rule of thumb:** widen / raise a range = harsher; narrow / lower it = gentler. Set a `prob` to `0` to
turn an effect off entirely. (e.g. *images too bright sometimes* -> lower the `1.4` in `brightness`.)

In [ ]:
DEGRADE = {
    # Low resolution: shrink to one of these pixel sizes then scale back to 224.
    # Smaller numbers = blurrier / more pixelated. (Always applied, one size per batch.)
    "downscale_sizes": [112, 128, 160, 192],

    # Gaussian blur (defocus). sigma = blur radius; higher = blurrier.
    "blur":       {"prob": 0.6, "sigma": (0.3, 2.0)},

    # Camera tilt (mounted-camera look). max_deg = max rotation (deg); shift = max sideways move (fraction of frame).
    "tilt":       {"max_deg": 15, "shift": 0.06},

    # Sensor noise. std on a 0-255 scale; higher = grainier.
    "noise":      {"prob": 0.6, "std": (3, 10)},

    # Brightness.  <1 = darker, >1 = brighter.   <<< too bright sometimes? lower the 1.4 >>>
    "brightness": {"prob": 0.6, "range": (0.7, 1.4)},

    # Contrast.  <1 = flatter/greyer, >1 = punchier.
    "contrast":   {"prob": 0.6, "range": (0.7, 1.4)},

    # Colour cast (cheap-camera white balance), per-channel gain. Closer to 1.0 = subtler tint.
    "color_cast": {"prob": 0.4, "range": (0.9, 1.12)},

    # Vignette (dark corners). Higher strength = darker corners.
    "vignette":   {"prob": 0.35, "strength": (0.15, 0.4)},

    # JPEG compression (applied at save). LOWER quality = more blocky artifacts.
    "jpeg":       {"prob": 0.6, "quality": (30, 70)},
}

## Degradation functions (GPU, batched)

One small function per effect - each reads its knobs from `DEGRADE`. `degrade_batch` just calls them in
order (capture -> lens -> sensor -> output). You normally won't need to touch these; tune `DEGRADE` instead.

In [ ]:
# --- small helpers ---
def _mask(B, prob, dev):
    """Per-sample on/off switch, shape [B,1,1,1]: 1 with probability `prob`, else 0."""
    return (torch.rand(B, 1, 1, 1, device=dev) < prob).float()

def _gauss_kernel(sigma, device, k=5):
    coords = torch.arange(k, device=device, dtype=torch.float32) - (k - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma * sigma)); g = g / g.sum()
    return (g[:, None] * g[None, :]).expand(3, 1, k, k).contiguous()

def _radial_r2(h, w, device):
    yy = torch.linspace(-1, 1, h, device=device)[:, None]
    xx = torch.linspace(-1, 1, w, device=device)[None, :]
    return ((xx ** 2 + yy ** 2) / 2.0).clamp(0, 1)        # normalized r^2 in [0,1]


# --- one function per effect (all read DEGRADE) ---
def low_resolution(x):
    """Shrink then upscale back -> the core low-res CCTV look (one size per batch)."""
    s = int(random.choice(DEGRADE["downscale_sizes"]))
    H, W = x.shape[-2:]
    x = F.interpolate(x, size=(s, s), mode="bilinear", align_corners=False)
    return F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)

def blur(x):
    """Gaussian defocus blur (one random sigma per batch)."""
    if random.random() >= DEGRADE["blur"]["prob"]:
        return x
    kern = _gauss_kernel(random.uniform(*DEGRADE["blur"]["sigma"]), x.device)
    return F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), kern, groups=3)

def camera_tilt(x):
    """Per-sample rotation + sideways shift; reflection border (no black corners)."""
    B = x.shape[0]; dev = x.device
    deg, shift = DEGRADE["tilt"]["max_deg"], DEGRADE["tilt"]["shift"]
    ang = (torch.rand(B, device=dev) * 2 - 1) * (deg * math.pi / 180)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1] = cos, -sin
    theta[:, 1, 0], theta[:, 1, 1] = sin, cos
    theta[:, 0, 2] = (torch.rand(B, device=dev) * 2 - 1) * shift
    theta[:, 1, 2] = (torch.rand(B, device=dev) * 2 - 1) * shift
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)

def sensor_noise(x):
    """Per-sample additive Gaussian noise (std on a 0-255 scale)."""
    B = x.shape[0]; dev = x.device
    lo, hi = DEGRADE["noise"]["std"]
    std = torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi) / 255.0
    return x + torch.randn_like(x) * std * _mask(B, DEGRADE["noise"]["prob"], dev)

def brightness_contrast(x):
    """Per-sample brightness, then contrast."""
    B = x.shape[0]; dev = x.device
    blo, bhi = DEGRADE["brightness"]["range"]
    mb = _mask(B, DEGRADE["brightness"]["prob"], dev)
    br = torch.empty(B, 1, 1, 1, device=dev).uniform_(blo, bhi)
    x = x * (br * mb + (1 - mb))                       # brightness 1.0 where the switch is off
    clo, chi = DEGRADE["contrast"]["range"]
    mc = _mask(B, DEGRADE["contrast"]["prob"], dev)
    ct = torch.empty(B, 1, 1, 1, device=dev).uniform_(clo, chi) * mc + (1 - mc)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    return (x - mean) * ct + mean

def colour_cast(x):
    """Per-sample, per-channel gain (cheap-camera white-balance tint)."""
    B = x.shape[0]; dev = x.device
    lo, hi = DEGRADE["color_cast"]["range"]
    m = _mask(B, DEGRADE["color_cast"]["prob"], dev)
    gain = torch.empty(B, 3, 1, 1, device=dev).uniform_(lo, hi)
    return x * (gain * m + (1 - m))

def vignette(x):
    """Per-sample corner darkening."""
    B, _, H, W = x.shape; dev = x.device
    lo, hi = DEGRADE["vignette"]["strength"]
    m = _mask(B, DEGRADE["vignette"]["prob"], dev)
    strength = torch.empty(B, 1, 1, 1, device=dev).uniform_(lo, hi)
    r2 = _radial_r2(H, W, dev)[None, None]
    return x * (1 - strength * m * r2)


def degrade_batch(x):
    """Apply every effect to a [B,3,H,W] batch in [0,1]. Edit the ORDER/SET of effects here if you want."""
    x = low_resolution(x)
    x = blur(x)
    x = camera_tilt(x)
    x = sensor_noise(x)
    x = brightness_contrast(x)
    x = colour_cast(x)
    x = vignette(x)
    return x.clamp(0, 1)


# --- I/O ---
def load_to_tensor(p, size=SIZE):
    img = Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

def save_img(t, path):
    arr = (t.clamp(0, 1) * 255).byte().cpu().numpy().transpose(1, 2, 0)
    lo, hi = DEGRADE["jpeg"]["quality"]
    q = random.randint(lo, hi) if random.random() < DEGRADE["jpeg"]["prob"] else 95
    Image.fromarray(arr).save(path, quality=q)

## Generate the degraded dataset (batched on the GPU)

2-5 degraded copies per source image; one fixed seed -> reproducible.

In [ ]:
from tqdm import tqdm

def generate_augmented_dataset():
    reset_degraded_folder()
    torch.manual_seed(utils.SEED); random.seed(utils.SEED); np.random.seed(utils.SEED)
    exts = (".jpg", ".jpeg", ".png")
    for cls in CLASSES:
        in_dir, out_dir = INPUT_DIR / cls, OUTPUT_DIR / cls
        srcs = sorted(p for p in in_dir.glob("*") if p.suffix.lower() in exts)
        work = [(p, i) for p in srcs for i in range(random.choice(NUM_AUG_CHOICES))]
        print(f"\n{cls}: {len(srcs)} sources -> {len(work)} degraded")
        for bs in tqdm(range(0, len(work), BATCH)):
            chunk = work[bs:bs + BATCH]
            x = torch.stack([load_to_tensor(p) for p, _ in chunk]).to(DEVICE)
            x = degrade_batch(x)
            for (p, i), t in zip(chunk, x):
                save_img(t, out_dir / f"{p.stem}_aug{i}.jpg")

generate_augmented_dataset()
print("\nDONE.")
for cls in CLASSES:
    print(cls, len(list((OUTPUT_DIR / cls).glob('*'))))

## Balance classes (upsample the smaller classes to the largest)

In [ ]:
def upsample_to_max():
    random.seed(utils.SEED); torch.manual_seed(utils.SEED)
    exts = (".jpg", ".jpeg", ".png")
    counts = {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES}
    target = max(counts.values())
    print("before:", counts, "| target:", target)
    for cls in CLASSES:
        srcs = sorted(p for p in (INPUT_DIR / cls).glob("*") if p.suffix.lower() in exts)
        need = target - counts[cls]
        if need <= 0 or not srcs:
            continue
        picks = [random.choice(srcs) for _ in range(need)]
        for bs in range(0, need, BATCH):
            chunk = picks[bs:bs + BATCH]
            x = degrade_batch(torch.stack([load_to_tensor(p) for p in chunk]).to(DEVICE))
            for j, (p, t) in enumerate(zip(chunk, x)):
                save_img(t, OUTPUT_DIR / cls / f"{p.stem}_extra{bs + j}.jpg")
    print("after :", {cls: len(list((OUTPUT_DIR / cls).glob('*'))) for cls in CLASSES})

upsample_to_max()

## Visual check: clean (top) vs degraded (bottom) per class

Run this after tuning `DEGRADE` to eyeball the effect before regenerating the whole set.

In [ ]:
import matplotlib.pyplot as plt
n = 8
fig, axes = plt.subplots(2 * len(CLASSES), n, figsize=(2 * n, 2 * 2 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    clean_files = sorted((INPUT_DIR / cls).glob('*'))[:n]
    for i, cp in enumerate(clean_files):
        axes[2*r, i].imshow(Image.open(cp)); axes[2*r, i].axis('off')
        augs = list((OUTPUT_DIR / cls).glob(f"{cp.stem}_aug*.jpg"))
        ax = axes[2*r+1, i]
        ax.imshow(Image.open(random.choice(augs)) if augs else Image.open(cp)); ax.axis('off')
    axes[2*r, 0].set_ylabel(cls, fontsize=12)
plt.tight_layout(); plt.show()